# Revised Emergency Readmissions TPMA - notes for Data Science team

As part of the NHP TPMA Review project we have reviewed the 'Emergency Readmissions within 28 Days' TPMA (IP-AA-028) and propose that this TPMA is replaced with 4 new TPMA:
- Emergency Readmissions within 30 Days following elective index spell.
- Emergency Readmissions within 30 Days following non-elective index spell for patients aged 0-17.
- Emergency Readmissions within 30 Days following non-elective index spell for patients aged 18-64.
- Emergency Readmissions within 30 Days following non-elective index spell for patients aged 65 plus.

#### What is 'Emergency Readmissions within 30 Days'

We noted differences between the 'Emergency Readmissions within 28 Days' TPMA and other common definitions/understanding of 'Emergency Readmissions' from other sources (for example fingertips <https://fingertips.phe.org.uk/search/readmissions>; <https://digital.nhs.uk/data-and-information/publications/statistical/compendium-emergency-readmissions/current/specifications/emergency-readmissions-within-30-days-of-discharge-from-hospital_5>). These are:

- Change to Emergency Readmission within 30 days (operationalised as 0-29 days inclusive).
- Exclude mental health related data.
- Exclude obstetrics related data.
- Exclude cancer related data.
- Exclude transfers.
- Discharge method for index spell should be on clinical advice.
- Exclude day case and regular admissions.

**Note:** At the handover meeting between project team and DS, it was also decided that the exclusion of well babies copied from the original TPMA code should be removed. This has not been accounted for in the code below.

For more information on this see:
- <https://github.com/The-Strategy-Unit/nhp_tpma_review/issues/8>.
- <https://github.com/The-Strategy-Unit/nhp_tpma_review/issues/11>.

#### Why the 4 subgroups?

We originally investigated splitting the TPMA by elective/non-elective index spell AND by three age groups (0-17, 18-64, 65plus). But found elective activity too small to split by age groups. So decided on having four subgroups:
- Elective index spell.
- Non elective index spell, aged 0-17.
- Non elective index spell, aged 18-64.
- Non elective index spell, aged 65plus.

For more information on this see <https://connect.strategyunitwm.nhs.uk/nhp_tpma_review/>.

# Code to make these 4 new TPMA

Below is code that can be used to make these 4 new TPMA. Attempts were made to make the code fit in with existing NHP model code, but tweaks and efficiencies may need to be made first. Some notes have been provided on this that may be helpful.

In [0]:
%pip install nhp_data

In [0]:
from pyspark.sql import functions as F
from nhp.data.get_spark import get_spark
from nhp.data.table_names import table_names
from nhp.data.inputs_data.ip.rates import get_population
from nhp.data.inputs_data.direct_standardisation import directly_standardise
from nhp.data.inputs_data.helpers import complete_age_sex_rows
from nhp.data.raw_data.mitigators.ip.hes_datasets import (
    any_diagnosis,
    nhp_apc,
    primary_diagnosis,
)
from pyspark.sql.window import Window

This is a function so I could apply the exclusions (MH, obstetrics, cancer, transfers, day cases) to both the index and readmission spell more easily. Notes:
- `left` was used in `filter` because I couldn't consistently get the `[ ]` notation to work in time.
- It may be worth considering if `cancer_spells` could be made once elsewhere and then called in rather than making it within the function.

In [0]:
def apply_readmissions_exclusions(data):
    data = data.alias("data")

    cancer_spells = (
        spark.read.table(table_names.hes_apc)
        .filter(F.col("last_episode_in_spell"))
        .join(spark.read.table(table_names.hes_apc_diagnoses), ["epikey", "fyear", "procode3"], "left")
        .filter(
            (F.col("diagnosis") == "Z511") |
            (F.left(F.col("diagnosis"), F.lit(1)) == "C") |
            (F.left(F.col("diagnosis"), F.lit(3)) == "D37") |
            (F.left(F.col("diagnosis"), F.lit(3)) == "D38") |
            (F.left(F.col("diagnosis"), F.lit(3)) == "D39") |          
            (F.left(F.col("diagnosis"), F.lit(2)) == "D4")             
            )
        .select(
            F.col("person_id_deid").alias("cancer_person_id"), 
            F.col("disdate").alias("cancer_disdate")
            )
        .distinct()
        .alias("cancer_spells")
        )

    filtered = (
        data
        # remove if cancer diagnosis in last 365 days
        .withColumn("365_days_before_disdate", F.date_add(F.col("disdate"), -365))
        .join(
            cancer_spells,
            (F.col("data.person_id") == F.col("cancer_spells.cancer_person_id")) &
            (F.col("cancer_spells.cancer_disdate") <= F.col("data.disdate")) &
            (F.col("cancer_spells.cancer_disdate") >= F.col("365_days_before_disdate")),
            "leftanti"
        )
        # remove obstetrics
        .filter(
            ~((F.col("tretspef") == "501") & F.col("primary_diagnosis").rlike("O")) &
            ~((F.col("tretspef") == "560") & F.col("primary_diagnosis").rlike("O")) & 
            ~((F.col("tretspef") == "610") & F.col("primary_diagnosis").rlike("O"))
            )
        # remove MH services
        .filter(~F.col("tretspef").isin("656", "700", "710", "711", "712", "713", "715", "720", "721", "722", "723", "724", "725", "726", "727", "730"))
        # remove transfers in and out
        .filter(~F.col("admimeth").isin("2B", "28", "81"))
        .filter(~F.col("disdest").isin("49", "50", "51", "52", "53", "84", "87"))
        # exclude day cases and regular attendances
        .filter(F.col("epitype") == 1)
        .filter(F.col("classpat") == 1)
        )
    return(filtered)

This is a function to get readmissions within 30 days for a point of delivery (POD) and age_group. Notes:
- I've tried to make this similar to <https://github.com/The-Strategy-Unit/nhp_data/blob/main/src/nhp/data/raw_data/mitigators/ip/activity_avoidance/readmission_within_28_days.py>.
- I had to join back to `hes_apc` to get the `disdest` column. But maybe the `disdest` column could be kept in the making of `nhp_apc` instead if feasible and beneficial.
- There are DQ issues with ages where patients had younger ages at readmission than in the index spell. A fix was made and has been explained in a comment.

In [0]:
def get_readmissions_within_30_days_pod(pod_filter, age_filter):
    spark = get_spark()

    readm = apply_readmissions_exclusions(
        nhp_apc
        # joining back to hes_apc to get disdest column - may be better to add disdest column to nhp_apc in longer run
        .join(
            spark.read.table(table_names.hes_apc).select(F.col("epikey"), F.col("fyear"), F.col("procode3"), F.col("disdest")), 
            ["epikey", "fyear", "procode3"],
            "left"
            )
        ).alias("readm")

    # make sure to use full hes table - our nhp views filter on certain columns
    # (e.g. not all providers included)
    prior = (
        spark.read.table(table_names.hes_apc)
        .filter(F.col("last_episode_in_spell"))
        # only spells discharged on clinical advice
        .filter(F.col("dismeth").isin(1, 3))
        # remove well babies
        .filter(F.col("well_baby_ind") == "N")
        .filter((F.col("sushrg") != "PB03Z") | F.col("sushrg").isNull())
        .filter(~((F.col("tretspef") == "424") & (F.col("epitype") == "3")))
        .withColumnRenamed("person_id_deid", "person_id")
    ) 

    # for exclusions need column for primary diagnoses
    df_primary_diagnosis = (
        spark.read.table(table_names.hes_apc_diagnoses)
        .filter(F.col("diag_order") == 1)
        .withColumnRenamed("diagnosis", "primary_diagnosis")
        )

    prior = (
        apply_readmissions_exclusions(
            prior
            .join(df_primary_diagnosis, ["epikey", "fyear", "procode3"], "left")
        )
        # filter for elective or non elective
        .filter(pod_filter)
        # filter to age group, applying the age logic from https://github.com/The-Strategy-Unit/nhp_data/blob/main/src/nhp/data/nhp_datasets/apc.py
        .withColumn(
            "age",
            F.when(
                (F.col("admiage") == 999) | F.col("admiage").isNull(),
                F.when(F.col("startage") > 7000, 0).otherwise(F.col("startage")),
            ).otherwise(F.col("admiage")),
        )
        .withColumn("age", F.when(F.col("age") > 90, 90).otherwise(F.col("age")))
        .filter(age_filter).alias("prior")
    )

    join_condition = [
        F.col("readm.person_id") == F.col("prior.person_id"),
        F.col("readm.epikey") != F.col("prior.epikey"),
        (F.col("readm.admidate") > F.col("prior.admidate"))
        | (F.col("readm.disdate") > F.col("prior.disdate")),
        F.datediff(F.col("readm.admidate"), F.col("prior.disdate")).between(0, 29),
        # because prior is filtered to age but there are DQ issues, have to make sure 
        # `readm.age` is within an acceptable range of `prior.age`, otherwise patient
        # is counted in multiple age groups. Since, looking at readmissions within 30
        # days, `reamd.age` should be between `prior.age` and `prior.age` plus the length of 
        # prior spell plus 1 year (to account for potential birthday during prior spell). But 
        # also there were many rows where the prior was 1 year more than the readmission and 
        # so those are kept too:
        F.col("readm.age").between(F.col("prior.age") - 1, F.col("prior.age") + F.col("prior.speldur") + 1) 
    ]

    return (
        readm.filter(F.col("admimeth").rlike("^2"))  
        .join(prior, join_condition, "semi")
        .select("fyear", "provider", "epikey")
        .withColumn("sample_rate", F.lit(1.0))
    )

This is where the conditions for age group and POD are set out: 

In [0]:
age_0_17 = F.col("age") < 18
age_18_64 = ((F.col("age") >= 18) & (F.col("age") < 65))
age_65_plus = F.col("age") >= 65
age_all = F.col("age") >= 0

non_elective = ((F.col("admimeth").rlike("^2")) & (F.col("classpat") == 1))
elective = ((F.col("admimeth").rlike("^1")) & (F.col("classpat") == 1))

This step was so I could easily reuse the functions in different contexts and may not be needed in the NHP model, but I've kept it in in case it's useful:

In [0]:
def get_readmissions_within_30_days_elective():
    return get_readmissions_within_30_days_pod(elective, age_all)

def get_readmissions_within_30_days_non_elective_0_17():
    return get_readmissions_within_30_days_pod(non_elective, age_0_17)
    
def get_readmissions_within_30_days_non_elective_18_64():
    return get_readmissions_within_30_days_pod(non_elective, age_18_64)
    
def get_readmissions_within_30_days_non_elective_65_plus():
    return get_readmissions_within_30_days_pod(non_elective, age_65_plus)

Checking the output:

In [0]:
get_readmissions_within_30_days_elective().limit(10).display()
#get_readmissions_within_30_days_non_elective_0_17().limit(10).display()
#get_readmissions_within_30_days_non_elective_18_64().limit(10).display()
#get_readmissions_within_30_days_non_elective_65_plus().limit(10).display()

In the NHP TPMA Review project, this data was then passed into functions to get the age and sex standardised rates that go into the TPMA explorer/`inputs_data` and to create similar outputs. This seemed to work suggesting that it might slot in nicely into parts of the existing NHP pipeline, but there may be slight differences and other changes will need to be made that we did not use (for example documentation), so we can't guarantee that the above code will slot in nicely to the whole NHP pipeline as is.